In [ ]:
# conda activate ethograph
import re
import natsort
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
import ethograph as eto
from movement.io import load_poses
from movement.kinematics import compute_velocity, compute_speed
from collections import defaultdict


In [ ]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime
from uuid import uuid4
import re

import numpy as np
import natsort
from dateutil.tz import tzlocal
from pynwb import NWBFile, NWBHDF5IO
from pynwb.image import ImageSeries

video_folder = Path("C:/Users/aksel/Desktop/POPPY_LABEL/20260308_01_Poppy")
pose_folder = Path("C:/Users/aksel/Desktop/POPPY_LABEL/raw/behav/dlc - Copy")
output_path = Path("C:/Users/aksel/Desktop/POPPY_LABEL/session.nwb")
camera_fps = 200.0

# ─── 1. Discover media files ───

media_by_trial: dict[int, dict[str, dict[str, Path]]] = defaultdict(
    lambda: defaultdict(dict)
)

patterns = {
    "video": (
        video_folder.glob("*.mp4"),
        re.compile(r"^2026-03-08_(?P<trial>\d{3})_Poppy-cam-(?P<camera>\d+)$"),
    ),
    "pose": (
        pose_folder.glob("*.csv"),
        re.compile(r"^2026-03-08_(?P<trial>\d{3})_Poppy-cam-(?P<camera>\d+)"),
    ),
}

for stream, (files, pattern) in patterns.items():
    for f in natsort.natsorted(files):
        m = pattern.search(f.stem)
        if m:
            trial = int(m["trial"])
            cam = f"cam_{m['camera']}"
            media_by_trial[trial][stream][cam] = f

# ─── 2. Derive labels ───

cam_labels = sorted({
    cam
    for trial_data in media_by_trial.values()
    for stream_data in trial_data.values()
    for cam in stream_data
})
stream_names = sorted(patterns.keys())
all_trials = sorted(media_by_trial.keys())

# ─── 3. Build NWB ───

nwbfile = NWBFile(
    session_description="NWB file for media alignment (ethograph generated).",
    identifier=str(uuid4()),
    session_start_time=datetime.now(tzlocal()),
)

# Trial columns
nwbfile.add_trial_column(name="trial_id", description="Original trial number")
for cam in cam_labels:
    for stream in stream_names:
        nwbfile.add_trial_column(
            name=f"{stream}_{cam}",
            description=f"{stream} filename for {cam}",
        )

# ImageSeries per camera (external file references with full paths)
for cam in cam_labels:
    nwbfile.create_device(name=cam, description=f"Behavior camera {cam}")
    external_files = [
        str(media_by_trial[t]["video"][cam])
        for t in all_trials
        if cam in media_by_trial[t].get("video", {})
    ]
    nwbfile.add_acquisition(
        ImageSeries(
            name=f"video_{cam}",
            description=f"Behavioral video from {cam}",
            external_file=external_files,
            format="external",
            starting_frame=np.zeros(len(external_files), dtype=np.int32),
            rate=camera_fps,
        )
    )

# Trial rows (filenames only — portable)
for trial in all_trials:
    row = {"trial_id": trial, "start_time": 0.0, "stop_time": 1.0}
    for cam in cam_labels:
        for stream in stream_names:
            path = media_by_trial[trial].get(stream, {}).get(cam)
            row[f"{stream}_{cam}"] = path.name if path else ""
    nwbfile.add_trial(**row)

# ─── 4. Save ───

with NWBHDF5IO(str(output_path), "w") as io:
    io.write(nwbfile)


nwbfile.trials

,start_time,stop_time,trial_id,pose_cam_1,video_cam_1,pose_cam_2,video_cam_2
id,,,,,,,
0,0.0,1.0,1,2026-03-08_001_Poppy-cam-1DLC_resnet50_Felix_cross_SessionsAug1shuffle1_200000_filtered.csv,2026-03-08_001_Poppy-cam-1.mp4,2026-03-08_001_Poppy-cam-2DLC_resnet50_Felix_cross_SessionsAug1shuffle1_200000_filtered.csv,2026-03-08_001_Poppy-cam-2.mp4
1,0.0,1.0,2,2026-03-08_002_Poppy-cam-1DLC_resnet50_Felix_cross_SessionsAug1shuffle1_200000_filtered.csv,2026-03-08_002_Poppy-cam-1.mp4,2026-03-08_002_Poppy-cam-2DLC_resnet50_Felix_cross_SessionsAug1shuffle1_200000_filtered.csv,2026-03-08_002_Poppy-cam-2.mp4
2,0.0,1.0,3,2026-03-08_003_Poppy-cam-1DLC_resnet50_Felix_cross_SessionsAug1shuffle1_200000_filtered.csv,2026-03-08_003_Poppy-cam-1.mp4,2026-03-08_003_Poppy-cam-2DLC_resnet50_Felix_cross_SessionsAug1shuffle1_200000_filtered.csv,2026-03-08_003_Poppy-cam-2.mp4
3,0.0,1.0,4,2026-03-08_004_Poppy-cam-1DLC_resnet50_Felix_cross_SessionsAug1shuffle1_200000_filtered.csv,2026-03-08_004_Poppy-cam-1.mp4,2026-03-08_004_Poppy-cam-2DLC_resnet50_Felix_cross_SessionsAug1shuffle1_200000_filtered.csv,2026-03-08_004_Poppy-cam-2.mp4


In [23]:
nwbfile.acquisition['video_cam_1']

Data type,uint8
Shape,"(0, 0, 0)"
Array size,0.00 bytes
Data type,int32
Shape,"(193,)"
Array size,772.00 bytes


In [19]:
nwbfile.acquisitions["pose_cam_1"]

AttributeError: 'NWBFile' object has no attribute 'acquisitions'

In [14]:
nwbfile.acquisition["video_cam_1"]

Data type,uint8
Shape,"(0, 0, 0)"
Array size,0.00 bytes
Data type,int32
Shape,"(193,)"
Array size,772.00 bytes


In [1]:
# ─── 3. Load trial datasets ───

# TODO: Replace with your actual dataset loading logic
ds_list = []
for trial in trials.trial_id:
    trial_pose_files = pose_by_trial.get(trial, [])
    if not trial_pose_files:
        continue

    # Load first camera/view (adjust if multi-camera)
    ds = load_poses.from_dlc_file(trial_pose_files[0], fps=30)
    ds["velocity"] = compute_velocity(ds.position)
    ds["speed"] = compute_speed(ds.position)
    ds.attrs["trial"] = trial
    for var in ds.data_vars:
        ds[var].attrs["type"] = "features"
    
    ds_list.append(ds)

NameError: name 'trials' is not defined

In [32]:
# ─── 4. Create TrialTree ───

dt = eto.from_datasets(ds_list)
# Inspect first trial of TrialTree
dt.itrial(0)

<xarray.DatasetView> Size: 985kB
Dimensions:      (time: 924, space: 2, keypoints: 22, individuals: 1)
Coordinates:
  * time         (time) float64 7kB 0.0 0.03333 0.06667 0.1 ... 30.7 30.73 30.77
  * space        (space) <U1 8B 'x' 'y'
  * keypoints    (keypoints) <U15 1kB 'beakTip' 'beakBase' ... 'desk1' 'desk2'
  * individuals  (individuals) <U12 48B 'individual_0'
Data variables:
    position     (time, space, keypoints, individuals) float64 325kB 159.2 .....
    confidence   (time, keypoints, individuals) float64 163kB 0.9964 ... 1.0
    velocity     (time, space, keypoints, individuals) float64 325kB 0.0 ... 0.0
    speed        (time, keypoints, individuals) float64 163kB 0.0 0.0 ... 33.78
Attributes:
    source_software:  DeepLabCut
    ds_type:          poses
    fps:              30.0
    time_unit:        seconds
    source_file:      C:/Users/aksel/Desktop/POPPY_LABEL/raw/behav/dlc - Copy...
    trial:            1

In [ ]:
# ─── 6. Set stream offsets (temporal alignment) ───

# No stream offsets configured

In [ ]:
# ─── 7. Save to NetCDF ───

dt.save("output.nc")  # TODO: set your output path